# Topic 10 — Solutions: Pivot Tables & MultiIndex

*Reference solutions for the objective tasks. Try the practice first.* The Warm-Up concept questions, Interview Lens and Reflection have **no key** — by design.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
products['list_price'] = np.where(products['list_price']<0, np.nan, products['list_price'])
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
orders['order_date'] = pd.to_datetime(orders['order_date'], format='mixed', dayfirst=True, errors='coerce')
orders['month'] = orders['order_date'].dt.to_period('M').astype(str)
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
master = (items.merge(products[['product_id','category','unit_cost']], on='product_id', how='left')
              .merge(orders[['order_id','channel','status','month','order_date']], on='order_id', how='left'))
master['line_cogs'] = master['quantity']*master['unit_cost']
master['line_profit'] = master['line_revenue'] - master['line_cogs']


### Warm-Up — NumPy recall (self-check)

In [ ]:
flat = np.arange(12)
grid = flat.reshape(3,4)
print(grid)

### Core lesson tasks

In [ ]:
ct = pd.crosstab(master['channel'], master['status'], normalize='index')
print(ct.round(3))

In [ ]:
pivot = master.pivot_table(index='channel', columns='month', values='line_revenue', aggfunc='sum', margins=True, fill_value=0)
print(pivot.iloc[:, :4])

In [ ]:
grp = master.groupby(['category','channel'])['line_revenue'].sum()
print(grp.unstack().round(0).head())

### Mixed review (earlier topics)

In [ ]:
months = sorted(master['month'].dropna().unique())
print(months[0], months[-1])

In [ ]:
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
print(len(master) == len(items))

### Data detective

In [ ]:
cat_month = master.pivot_table(index='category', columns='month', values='line_revenue', aggfunc='sum', fill_value=0)
print(cat_month.sum(axis=1).sort_values(ascending=False))

### Interview Lens — discussion points (not full answers)
- **Q15:** pivot_table for a ready-to-read 2-D grid/heatmap; groupby for multi-metric aggregation or further computation. pivot_table = groupby + unstack.